# PhotoPrism MLOps — Full Provisioning & Bring-Up

End-to-end provisioner. Run all cells top to bottom for a fresh lease.

**What this notebook does:**
1. Reserve a 24-hour KVM@TACC lease
2. Install Terraform, provision 3 VMs + network + floating IP
3. Print commands to run on your Mac and on node1

**Prerequisites (already in /work):**
- `clouds.yaml` — KVM@TACC application credentials
- SSH key `id_rsa_chameleon` registered on Chameleon
- On your Mac:  `~/.ssh/id_rsa_chameleon`, `~/.ssh/id_proj24_gpu`, `~/.ssh/sealed-secrets-key-backup.yaml`

**Teardown is at the bottom — uncomment and run.**

## Step 1: Configure Chameleon Context

In [ ]:
from chi import server, context, lease, network
import chi, os, time, datetime, subprocess, shutil, json

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

# ---- Project configuration ----
PROJECT_PREFIX = "proj24"
SSH_KEY_NAME   = "id_rsa_chameleon"
LEASE_END      = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(hours=24)
VM_FLAVOR      = "m1.xlarge"
VM_COUNT       = 3
VM_IMAGE       = "CC-Ubuntu24.04"

print(f"Lease will end: {LEASE_END.isoformat()}")

## Step 2: Install Terraform

In [ ]:
TF_VERSION = "1.14.4"

commands = [
    "mkdir -p /work/.local/bin",
    f"wget -q https://releases.hashicorp.com/terraform/{TF_VERSION}/terraform_{TF_VERSION}_linux_amd64.zip",
    f"unzip -o -q terraform_{TF_VERSION}_linux_amd64.zip",
    "mv terraform /work/.local/bin",
    f"rm terraform_{TF_VERSION}_linux_amd64.zip",
]

for cmd in commands:
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error: {cmd}\n{result.stderr}")
    else:
        print(f"Done: {cmd}")

os.environ["PATH"] = "/work/.local/bin:" + os.environ["PATH"]

result = subprocess.run("terraform --version", shell=True, capture_output=True, text=True)
print(result.stdout.split('\n')[0])

## Step 3: Create Lease

In [ ]:
LEASE_NAME = f"lease-infra-{PROJECT_PREFIX}"

l = lease.Lease(
    LEASE_NAME,
    end_date=LEASE_END,
)
l.add_flavor_reservation(
    id=chi.server.get_flavor_id(VM_FLAVOR),
    amount=VM_COUNT,
)
l.submit(idempotent=True)

reservation_id = l.get_reserved_flavors()[0].id
print(f"Lease: {LEASE_NAME} — Status: ACTIVE")
print(f"Reservation flavor ID: {reservation_id}")

## Step 4: Set Up Terraform Files

Creates the Terraform configuration adapted from the course lab IaC repo (`gourmetgram-iac`).

**Resources created by Terraform:**
- Private network + subnet (192.168.1.0/24, no gateway)
- 3 ports on private network (fixed IPs, no port security)
- 3 ports on sharednet1 (with security groups)
- 3 compute instances (CC-Ubuntu24.04, m1.medium)
- 1 floating IP assigned to node1

In [ ]:
# Cell: Clone infra repo and set up Terraform
import subprocess, shutil

INFRA_REPO = "https://github.com/akashchauhanweb/photoprism-mlops-infra.git"
infra_dir = "/work/photoprism-mlops-infra"
tf_dir = f"{infra_dir}/tf/kvm"

# Clone or pull the repo
if os.path.exists(infra_dir):
    result = subprocess.run("git pull", shell=True, cwd=infra_dir, capture_output=True, text=True)
    print(f"Repo updated: {result.stdout.strip()}")
else:
    result = subprocess.run(f"git clone {INFRA_REPO} {infra_dir}", shell=True, capture_output=True, text=True)
    print(f"Repo cloned: {result.stdout.strip()}")

# Copy clouds.yaml into tf directory (secret — not in repo)
shutil.copy("/work/clouds.yaml", f"{tf_dir}/clouds.yaml")
print(f"clouds.yaml copied to {tf_dir}")

# Verify Terraform files exist
import os
tf_files = [f for f in os.listdir(tf_dir) if f.endswith('.tf')]
print(f"Terraform files found: {', '.join(sorted(tf_files))}")

## Step 5: Terraform Init, Plan, and Apply

In [ ]:
# Build a clean environment — remove Chameleon Jupyter's OS_ vars
# which conflict with our clouds.yaml (they point to CHI@UC)
clean_env = {k: v for k, v in os.environ.items() if not k.startswith("OS_")}
clean_env["OS_CLOUD"]        = "openstack"
clean_env["PATH"]            = "/work/.local/bin:" + clean_env.get("PATH", "")
clean_env["HOME"]            = os.environ.get("HOME", "/home/jovyan")
clean_env["TF_VAR_suffix"]   = PROJECT_PREFIX
clean_env["TF_VAR_key"]      = SSH_KEY_NAME
clean_env["TF_VAR_reservation"] = reservation_id

def run_tf(command, description):
    print(f"\n{'='*60}")
    print(f"  {description}")
    print(f"{'='*60}")
    result = subprocess.run(
        command, shell=True, cwd=tf_dir, env=clean_env,
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise Exception(f"{description} failed with return code {result.returncode}")
    return result.returncode

run_tf("terraform init", "Terraform Init")
run_tf("terraform validate", "Terraform Validate")
run_tf("terraform plan", "Terraform Plan")

In [ ]:
# Ensure all required security groups exist on the KVM@TACC project.
# Terraform uses `data` sources for these; if any is missing, `terraform plan` 404s.
REQUIRED_SGS = {
    "allow-ssh":          22,
    "allow-http-80":      80,
    "allow-2342":         2342,
    "allow-8000":         8000,
    "allow-8080":         8080,
    "allow-9001":         9001,
    "allow-6333":         6333,
    "allow-30234":        30234,
    "allow-30500":        30500,
    "allow-30633-proj24": 30633,
    "allow-30443":        30443,
    "allow-30300":        30300,
    "allow-30810":        30810,
}

def _os(cmd):
    r = subprocess.run(cmd, shell=True, env=clean_env,
                       capture_output=True, text=True)
    return r.returncode, r.stdout.strip(), r.stderr.strip()

rc, existing, _ = _os("openstack security group list -f value -c Name")
existing = set(existing.splitlines())
for name, port in REQUIRED_SGS.items():
    if name in existing:
        print(f"  [ok]      {name}")
        continue
    print(f"  [create]  {name}  (tcp/{port})")
    _os(f"openstack security group create {name} --description 'auto: proj24 bringup'")
    _os(f"openstack security group rule create --protocol tcp "
        f"--dst-port {port} --remote-ip 0.0.0.0/0 {name}")
print("\nAll required security groups present.")


In [ ]:
# Apply — creates all resources
# If Terraform 409s on port creation: stale DOWN ports linger from a
# previous lease. Fix by running from this notebook:
#   subprocess.run("openstack port list --status DOWN -f value -c ID "
#                  "| xargs -r -n1 openstack port delete",
#                  shell=True, env=clean_env)
# then re-run terraform apply.
run_tf("terraform apply -auto-approve", "Terraform Apply")

# Extract outputs
result = subprocess.run(
    "terraform output -json",
    shell=True, cwd=tf_dir, env=clean_env,
    capture_output=True, text=True
)
outputs = json.loads(result.stdout)
floating_ip = outputs["floating_ip"]["value"]

print(f"\n{'='*72}")
print(f"  Infrastructure provisioned successfully!")
print(f"{'='*72}")
print(f"\n  Floating IP: {floating_ip}\n")

print(f"{'='*72}")
print(f"  NEXT STEPS")
print(f"{'='*72}\n")

print("STEP A — From your Mac, copy SSH keys and sealed-secrets backup to node1:")
print(f"   scp -i ~/.ssh/id_rsa_chameleon \\")
print(f"       ~/.ssh/id_rsa_chameleon \\")
print(f"       ~/.ssh/id_proj24_gpu \\")
print(f"       ~/.ssh/sealed-secrets-key-backup.yaml \\")
print(f"       cc@{floating_ip}:~/.ssh/\n")

print("STEP B — Copy clouds.yaml for openstack CLI (create dir first):")
print(f"   ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip} 'mkdir -p ~/.config/openstack'")
print(f"   scp -i ~/.ssh/id_rsa_chameleon \\")
print(f"       ~/Documents/photoprism-mlops-infra/clouds.yaml \\")
print(f"       cc@{floating_ip}:~/.config/openstack/clouds.yaml\n")
print("   (adjust the source path to wherever your clouds.yaml actually lives)\n")

print("STEP C — SSH to node1, disable broken IPv6, set up kubespray (~20 min):\n")
print(f"   ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
print("")
print("   # On node1:")
print("   chmod 600 ~/.ssh/id_rsa_chameleon ~/.ssh/id_proj24_gpu ~/.ssh/sealed-secrets-key-backup.yaml")
print("")
print("   # Disable IPv6 on ens3 (Chameleon's IPv6 doesn't route; Go/apt/CoreDNS hang)")
print("   # Do this on ALL nodes via SSH from node1:")
print("   for ip in 192.168.1.11 192.168.1.12 192.168.1.13; do")
print("       ssh -o StrictHostKeyChecking=no cc@$ip 'sudo sysctl -w net.ipv6.conf.ens3.disable_ipv6=1'")
print("   done")
print("")
print("   sudo apt update && sudo apt install -y python3-pip python3-venv git tmux")
print("   git clone -b release-2.26 https://github.com/kubernetes-sigs/kubespray.git")
print("   cd kubespray")
print("   python3 -m venv .venv && source .venv/bin/activate")
print("   pip install -r requirements.txt ruamel.yaml")
print("")
print("   # Build inventory")
print("   cp -rfp inventory/sample inventory/mycluster")
print("   CONFIG_FILE=inventory/mycluster/hosts.yaml \\")
print("       python3 contrib/inventory_builder/inventory.py 192.168.1.11 192.168.1.12 192.168.1.13")
print("")
print("   # Single control plane: strip node2/node3 from kube_control_plane")
print("   python3 -c \"")
print("import sys, yaml")
print("p='inventory/mycluster/hosts.yaml'")
print("d=yaml.safe_load(open(p))")
print("d['all']['children']['kube_control_plane']['hosts']={'node1': None}")
print("open(p,'w').write(yaml.safe_dump(d, default_flow_style=False))")
print("\"")
print("")
print("   # Set ansible user, SSH key, and disable IPv6 DNS")
print("   cat > inventory/mycluster/group_vars/all/all.yml <<'EOF'")
print("   ansible_user: cc")
print("   ansible_ssh_private_key_file: /home/cc/.ssh/id_rsa_chameleon")
print("   ansible_become: true")
print("   ansible_become_method: sudo")
print("   disable_ipv6_dns: true")
print("   EOF")
print("")
print("   # Enable helm flag (note: kubespray often skips the actual binary;")
print("   # Step D installs it again via get-helm-3 to be safe)")
print("   sed -i 's/^helm_enabled: false/helm_enabled: true/' \\")
print("       inventory/mycluster/group_vars/k8s_cluster/addons.yml")
print("")
print("   # Test connectivity")
print("   ansible -i inventory/mycluster/hosts.yaml all -m ping")
print("")
print("   # Run cluster setup (in tmux so SSH drops don't kill it)")
print("   tmux new -s kubespray")
print("   cd ~/kubespray && source .venv/bin/activate")
print("   ansible-playbook -i inventory/mycluster/hosts.yaml cluster.yml -b 2>&1 | tee /tmp/kubespray.log")
print("   # Detach: Ctrl+B then D. Reattach: tmux attach -t kubespray")
print("")

print("STEP D — Post-kubespray cluster setup:\n")
print("   # Make kubectl work as cc user")
print("   mkdir -p ~/.kube && sudo cp /etc/kubernetes/admin.conf ~/.kube/config")
print("   sudo chown $(id -u):$(id -g) ~/.kube/config")
print("")
print("   # Install kubeseal CLI")
print(f"   curl -L -o /tmp/kubeseal.tar.gz https://github.com/bitnami-labs/sealed-secrets/releases/download/v0.27.1/kubeseal-0.27.1-linux-amd64.tar.gz")
print("   tar -xzf /tmp/kubeseal.tar.gz -C /tmp kubeseal")
print("   sudo install -m 0755 /tmp/kubeseal /usr/local/bin/kubeseal")
print("")
print("   # Install helm")
print("   curl https://raw.githubusercontent.com/helm/helm/main/scripts/get-helm-3 | bash")
print("")
print("   # Install local-path-provisioner (for PVCs on bare VMs)")
print("   kubectl apply -f https://raw.githubusercontent.com/rancher/local-path-provisioner/master/deploy/local-path-storage.yaml")
print("   kubectl patch storageclass local-path -p \\")
print("     '{\"metadata\":{\"annotations\":{\"storageclass.kubernetes.io/is-default-class\":\"true\"}}}'")
print("")
print("   # Install metrics-server (needs --kubelet-insecure-tls on Chameleon)")
print("   kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml")
print("   kubectl -n kube-system patch deployment metrics-server --type='json' \\")
print("     -p='[{\"op\":\"add\",\"path\":\"/spec/template/spec/containers/0/args/-\",\"value\":\"--kubelet-insecure-tls\"}]'")
print("")
print("   # Patch CoreDNS to use external DNS (Chameleon upstream DNS is flaky)")
print("   kubectl -n kube-system get configmap coredns -o yaml | \\")
print("     sed 's|forward . /etc/resolv.conf|forward . 8.8.8.8 1.1.1.1|' | \\")
print("     kubectl apply -f -")
print("   kubectl -n kube-system rollout restart deployment coredns\n")

print("STEP E — Install Sealed Secrets + Loki/Grafana:\n")
print("   kubectl apply -f https://github.com/bitnami-labs/sealed-secrets/releases/download/v0.27.1/controller.yaml")
print("")
print("   helm repo add grafana https://grafana.github.io/helm-charts")
print("   helm repo update")
print("   helm upgrade --install loki-stack grafana/loki-stack \\")
print("     --namespace monitoring --create-namespace \\")
print("     --set grafana.enabled=true \\")
print("     --set grafana.service.type=NodePort \\")
print("     --set grafana.service.nodePort=30300\n")

print("STEP F — Clone infra repo and run bring_up.sh:")
print("   git clone https://github.com/akashchauhanweb/photoprism-mlops-infra.git")
print("   cd photoprism-mlops-infra")
print("   cp scripts/config.env.example scripts/config.env")
print("   bash scripts/bring_up.sh")
print("")
print("   # bring_up.sh prints all service URLs + credentials at the end.")


---

## Teardown

Run these cells **only** when you want to destroy all infrastructure and free resources.

In [ ]:
# TEARDOWN: Destroy all Terraform-managed resources
# run_tf("terraform destroy -auto-approve", "Terraform Destroy")

In [ ]:
# TEARDOWN: Delete the lease
# l.delete()
# print("Lease deleted.")